<a href="https://colab.research.google.com/github/PavikaSharma15/Generic-Fraud-Detection-URT-XFraud-/blob/main/URT_XFraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from collections import defaultdict, Counter

df = pd.read_csv("/content/synthetic_health_claims.csv")


generic adapter

In [2]:
class GenericFraudAdapter:
    def __init__(self, df):
        self.df = df.copy()

    def adapt(self):
        out = pd.DataFrame()

        # amount
        for c in self.df.columns:
            if c.lower() in ["amount", "claim_amount", "transaction_amount"]:
                out["amount"] = self.df[c]
                break

        # time
        for c in self.df.columns:
            if c.lower() in ["time", "timestamp", "claim_time"]:
                out["time"] = pd.to_datetime(self.df[c])
                break

        # user
        for c in self.df.columns:
            if c.lower() in ["user_id", "customer_id", "policy_holder"]:
                out["user_id"] = self.df[c].astype(str)
                break
        else:
            out["user_id"] = self.df.index.astype(str)

        # endpoint / village / agent
        for c in self.df.columns:
            if c.lower() in ["village", "agent_id", "hospital", "endpoint"]:
                out["endpoint"] = self.df[c].astype(str)
                break
        else:
            out["endpoint"] = "UNKNOWN"

        # return out.fillna(0)
        out = out.fillna(0)
        out["endpoint_entity"] = out["endpoint"]
        return out


profile builder

In [3]:
class UserProfileBuilder:
    def __init__(self, df):
        self.df = df

    def build(self):
        stats = self.df.groupby("user_id")["amount"].agg(
            mean_amount="mean",
            std_amount="std",
            max_amount="max"
        ).fillna(0)
        return stats


temporal cluster analysis

In [4]:
class TemporalClusterAnalyzer:
    def __init__(self, df, window=5):
        self.df = df.reset_index(drop=True)
        self.window = window

    def score(self, idx):
        # index-based sliding window
        start = max(0, idx - self.window)
        end = min(len(self.df), idx + self.window + 1)

        subset = self.df.iloc[start:end]

        # same endpoint/entity appearing multiple times nearby
        if "endpoint_entity" in self.df.columns:
            counts = subset["endpoint_entity"].value_counts()
            val = self.df.iloc[idx]["endpoint_entity"]
            #return counts.get(val, 0) / self.window
            raw = counts.get(val, 0)
            score = raw / self.window
            return min(score, 1.0)

        return 0.0


Isolation Forest (UNSUPERVISED RISK SIGNAL)

In [5]:
class RiskEngine:
    def __init__(self):
        self.model = IsolationForest(
            n_estimators=100,
            contamination=0.05,
            random_state=42
        )

    def fit(self, X):
        self.model.fit(X)

    def score(self, X):
        raw = -self.model.decision_function(X)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-6)


In [6]:
# =========================
# EXPLANATION + BELIEF
# =========================

class ExplanationEngine:
    def explain(self, amount_ratio, cluster_score, risk_score):
        exps = []

        # 🔹 Amount spike (user-relative)
        if amount_ratio > 3.0:
            exps.append(("amount_spike", +1, 0.6))

        # 🔹 Shared endpoint / village / agent
        if cluster_score > 0.8:
            exps.append(("shared_endpoint", +1, 0.6))

        # 🔹 Model itself is confident it's risky
        if risk_score > 0.7:
            exps.append(("model_anomaly", +1, 0.7))

        # 🔹 Normal behavior fallback
        if not exps:
            exps.append(("normal_behavior", -1, 0.4))

        return exps


def explanation_belief(explanations):
    """
    Converts explanations → soft belief score [0,1]
    """
    score = 0.0
    for _, direction, strength in explanations:
        score += direction * strength

    return 1 / (1 + np.exp(-score))


# =========================
# USER-RELATIVE DECISION
# =========================

def decide_user_relative(
    amount_ratio,
    user_mean,
    user_std,
    cluster_score,
    risk_score,
    belief
):
    volatility = user_std / (user_mean + 1e-6)

    review_threshold = 2.8 + volatility
    block_threshold = 4.5 + volatility

    # 🚫 BLOCK (strong signals together)
    if (
        amount_ratio > block_threshold
        and cluster_score > 0.9
        and risk_score > 0.8
    ):
        return "BLOCK"

    # ⚠️ MANUAL REVIEW (any 2 signals)
    signals = 0
    signals += amount_ratio > review_threshold
    signals += cluster_score > 0.9
    signals += belief > 0.7
    signals += risk_score > 0.75

    if signals >= 2:
        return "MANUAL_REVIEW"

    return "APPROVE"


# =========================
# FINAL PIPELINE (CRASH SAFE)
# =========================

def run_pipeline(df, n_rows=40):
    base = GenericFraudAdapter(df).adapt()
    base = base.iloc[:n_rows].copy()

    if "time" not in base.columns:
        base["time"] = np.arange(len(base))

    profiles = UserProfileBuilder(base).build()
    clusterer = TemporalClusterAnalyzer(base)

    risk_engine = RiskEngine()
    X = base[["amount"]]
    risk_engine.fit(X)
    risks = risk_engine.score(X)

    explainer = ExplanationEngine()
    outputs = []

    for i, row in base.iterrows():
        prof = profiles.loc[row["user_id"]]

        amount_ratio = row["amount"] / (prof["mean_amount"] + 1e-6)
        cluster_score = clusterer.score(i)

        exps = explainer.explain(
            amount_ratio,
            cluster_score,
            risks[i]
        )

        belief = explanation_belief(exps)

        decision = decide_user_relative(
            amount_ratio,
            prof["mean_amount"],
            prof["std_amount"],
            cluster_score,
            risks[i],
            belief
        )

        outputs.append({
            "decision": decision,

            "belief": float(belief),
            "risk": float(risks[i]),
            "amount_ratio": float(amount_ratio),
            "cluster_score": float(cluster_score),
            "explanations": exps
        })

    return outputs


In [7]:
outputs = run_pipeline(df, n_rows=40)

for o in outputs:
    print(o)

print("Summary:", Counter(o["decision"] for o in outputs))


{'decision': 'APPROVE', 'belief': 0.6456563062257954, 'risk': 0.2453800530949986, 'amount_ratio': 0.9999999999978371, 'cluster_score': 1.0, 'explanations': [('shared_endpoint', 1, 0.6)]}
{'decision': 'APPROVE', 'belief': 0.6456563062257954, 'risk': 0.5035432663664213, 'amount_ratio': 0.9999999999994691, 'cluster_score': 1.0, 'explanations': [('shared_endpoint', 1, 0.6)]}
{'decision': 'APPROVE', 'belief': 0.6456563062257954, 'risk': 0.4142477800238191, 'amount_ratio': 0.9999999999993336, 'cluster_score': 1.0, 'explanations': [('shared_endpoint', 1, 0.6)]}
{'decision': 'APPROVE', 'belief': 0.6456563062257954, 'risk': 0.14511334050867739, 'amount_ratio': 0.9999999999982525, 'cluster_score': 1.0, 'explanations': [('shared_endpoint', 1, 0.6)]}
{'decision': 'APPROVE', 'belief': 0.6456563062257954, 'risk': 0.36702036011162686, 'amount_ratio': 0.9999999999995212, 'cluster_score': 1.0, 'explanations': [('shared_endpoint', 1, 0.6)]}
{'decision': 'APPROVE', 'belief': 0.6456563062257954, 'risk': 0